# Results Analysis — Sine Wave Denoising

This notebook is the final comparison report for the three denoising
architectures (MLP, RNN, LSTM) trained on a noisy mixture of four sine
waves. It reproduces the items called out in PRD §A5:

- **G2** — test-set MSE per model with mean ± std over ≥3 seeds.
- **G3** — hyperparameter sweep plots (≥3 axes per model).
- **F17** — per-component MSE breakdown.
- **F18** — noise-robustness curve: test MSE as a function of σ.
- **F19** — qualitative predicted-vs-clean window plots per component.

To keep the notebook quick to run on a laptop, it uses a smaller
in-notebook config (shorter signal, fewer epochs) than `config/default.json`.
The full reproducible numbers live under `runs/` after running the
training CLI; this notebook is the *methodology* and *visual* report.


In [ ]:
from __future__ import annotations

import copy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from sine_denoiser import SDK
from sine_denoiser.evaluation.robustness import sweep_noise
from sine_denoiser.plotting.predictions import plot_predictions
from sine_denoiser.plotting.sweep import plot_sweep_line

MODEL_NAMES = ("mlp", "rnn", "lstm")
NOTEBOOK_OUT = Path("runs/notebook")
NOTEBOOK_OUT.mkdir(parents=True, exist_ok=True)


## 1. Notebook configuration

A lightweight configuration that runs the entire pipeline end-to-end without
requiring GPU resources. Its structure matches `config/default.json` allowing any
section to be easily replaced with the production configuration.

In [ ]:
BASE_CONFIG: dict = {
    "version": "1.00",
    "data": {
        "num_components": 4,
        "duration_s": 1.0,
        "sample_rate_hz": 1000,
        "noise_sigma": 0.5,
        "frequencies_hz": [2.0, 5.0, 11.0, 17.0],
        "phases_rad": [0.0, 0.7, 1.5, 2.3],
        "amplitudes": [1.0, 1.0, 1.0, 1.0],
        "context_window": 10,
        "split": {"train": 0.8, "val": 0.1, "test": 0.1},
        "data_seed": 0,
    },
    "model": {
        "mlp": {"hidden_size": 32, "num_layers": 2, "activation": "relu"},
        "rnn": {"hidden_size": 32, "num_layers": 1, "nonlinearity": "tanh"},
        "lstm": {"hidden_size": 32, "num_layers": 1},
    },
    "training": {
        "optimizer": "adam",
        "lr": 1e-3,
        "batch_size": 128,
        "epochs": 5,
        "early_stopping_patience": 3,
        "seeds": [0, 1, 2],
    },
}
SEEDS = BASE_CONFIG["training"]["seeds"]


## 2. Data preview

Generate the pure components and the noisy mixed signal once, and plot
the first second so the noise level is visible.

In [ ]:
sdk_preview = SDK(config=BASE_CONFIG)
bundle = sdk_preview.generate_data()
print(f"pure shape:  {bundle.pure.shape}")
print(f"mixed shape: {bundle.mixed.shape}")

t = np.arange(bundle.mixed.shape[0]) / BASE_CONFIG["data"]["sample_rate_hz"]
fig, axes = plt.subplots(2, 1, figsize=(8.0, 4.5), sharex=True)
for k in range(bundle.pure.shape[0]):
    axes[0].plot(t, bundle.pure[k], label=f"component {k}", alpha=0.8)
axes[0].set_title("Pure components")
axes[0].set_ylabel("amplitude")
axes[0].legend(loc="upper right", ncol=2, fontsize=8)
axes[0].grid(True, alpha=0.3)
axes[1].plot(t, bundle.mixed, color="black")
axes[1].set_title(f"Noisy mixed signal (σ={BASE_CONFIG['data']['noise_sigma']})")
axes[1].set_xlabel("t [s]")
axes[1].set_ylabel("amplitude")
axes[1].grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## 3. G2 — Test MSE per model, mean ± std over ≥3 seeds

Each model is trained from scratch for every seed. After training,
the final test MSE from each run is collected and summarized using the mean ± standard deviation across all seeds.
The trained models from the *first* seed are later reused for
qualitative visualizations and the noise-robustness analysis.

In [ ]:
def train_seed(model_name: str, seed: int, cfg: dict):
    sdk = SDK(config=cfg)
    run = sdk.train(model_name, seed=seed)
    report = sdk.evaluate(run)
    return sdk, run, report


seed_runs: dict[str, list] = {name: [] for name in MODEL_NAMES}
seed_reports: dict[str, list] = {name: [] for name in MODEL_NAMES}
first_seed_sdks: dict[str, SDK] = {}
first_seed_runs: dict = {}

for name in MODEL_NAMES:
    for seed in SEEDS:
        sdk, run, report = train_seed(name, seed, BASE_CONFIG)
        seed_runs[name].append(run)
        seed_reports[name].append(report)
        if seed == SEEDS[0]:
            first_seed_sdks[name] = sdk
            first_seed_runs[name] = run
    mses = [r.test_mse for r in seed_reports[name]]
    print(f"{name:>4}: test MSE per seed = "
          f"{['%.4f' % v for v in mses]}")


In [ ]:
g2_summary = {}
for name in MODEL_NAMES:
    mses = np.array([r.test_mse for r in seed_reports[name]], dtype=np.float64)
    g2_summary[name] = (float(mses.mean()), float(mses.std(ddof=0)))

print("Test MSE — mean ± std over seeds")
print("-" * 40)
for name, (m, s) in g2_summary.items():
    print(f"  {name:>4}: {m:.4f} ± {s:.4f}  (n={len(SEEDS)})")

fig, ax = plt.subplots(figsize=(5.0, 3.5))
xs = np.arange(len(MODEL_NAMES))
means = [g2_summary[n][0] for n in MODEL_NAMES]
stds = [g2_summary[n][1] for n in MODEL_NAMES]
ax.bar(xs, means, yerr=stds, capsize=6, color=["C0", "C1", "C2"])
ax.set_xticks(xs)
ax.set_xticklabels([n.upper() for n in MODEL_NAMES])
ax.set_ylabel("test MSE")
ax.set_title(f"G2: test MSE mean ± std (n={len(SEEDS)} seeds)")
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
plt.show()


## 4. F17 — Per-component MSE breakdown

The evaluator computes the MSE separately for each target component
`c ∈ {0..3}`. We average across seeds and visualise the per-component
profile per model — this exposes whether a given architecture
struggles on a specific frequency.

In [ ]:
num_components = BASE_CONFIG["data"]["num_components"]
per_comp = {
    name: np.stack([
        np.array(r.test_mse_per_component, dtype=np.float64)
        for r in seed_reports[name]
    ])
    for name in MODEL_NAMES
}
per_comp_mean = {name: per_comp[name].mean(axis=0) for name in MODEL_NAMES}
per_comp_std = {name: per_comp[name].std(axis=0, ddof=0) for name in MODEL_NAMES}

print("Per-component test MSE — mean over seeds")
header = "  c   " + "   ".join(f"{n:>10}" for n in MODEL_NAMES)
print(header)
print("-" * len(header))
for k in range(num_components):
    row = "  " + str(k) + "   " + "   ".join(
        f"{per_comp_mean[n][k]:>10.4f}" for n in MODEL_NAMES
    )
    print(row)

fig, ax = plt.subplots(figsize=(7.0, 4.0))
width = 0.25
xs = np.arange(num_components)
for i, name in enumerate(MODEL_NAMES):
    ax.bar(
        xs + (i - 1) * width,
        per_comp_mean[name],
        width=width,
        yerr=per_comp_std[name],
        capsize=4,
        label=name.upper(),
    )
ax.set_xticks(xs)
ax.set_xticklabels([f"c={k}" for k in range(num_components)])
ax.set_ylabel("test MSE")
ax.set_title("F17: per-component test MSE (mean ± std over seeds)")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
plt.show()


## 5. F19 — Qualitative predicted-vs-clean plots

For the first-seed checkpoint of each model, draw a handful of test
windows per component and overlay the predicted curve onto the clean
ground truth. Plots are written under `runs/notebook/predictions/` via
`plotting.predictions.plot_predictions`, then displayed inline.

In [ ]:
pred_root = NOTEBOOK_OUT / "predictions"
pred_root.mkdir(parents=True, exist_ok=True)

prediction_paths: dict[str, list[Path]] = {}
for name in MODEL_NAMES:
    sdk = first_seed_sdks[name]
    run = first_seed_runs[name]
    out_dir = pred_root / name
    paths = plot_predictions(
        run.model,
        sdk.generate_data().loaders.test,
        out_dir=out_dir,
        num_components=num_components,
        model_name=name,
        examples_per_component=4,
    )
    prediction_paths[name] = paths
    print(f"{name}: wrote {len(paths)} component PNG(s) to {out_dir}")

fig, axes = plt.subplots(
    len(MODEL_NAMES), num_components, figsize=(3.6 * num_components, 2.6 * len(MODEL_NAMES)),
    squeeze=False,
)
for row, name in enumerate(MODEL_NAMES):
    sdk = first_seed_sdks[name]
    run = first_seed_runs[name]
    test_loader = sdk.generate_data().loaders.test
    examples_collected = {k: None for k in range(num_components)}
    for x_np, c_np, y_np in test_loader:
        for i, k in enumerate(c_np):
            k_int = int(k)
            if examples_collected[k_int] is None:
                examples_collected[k_int] = (x_np[i], int(k), y_np[i])
        if all(v is not None for v in examples_collected.values()):
            break
    for col in range(num_components):
        ax = axes[row][col]
        x_ctx, c_val, y_clean = examples_collected[col]
        y_hat = sdk.predict(run, x_ctx, c_val).cpu().numpy().squeeze()
        xs = np.arange(y_clean.shape[0])
        ax.plot(xs, y_clean, marker="o", color="C0", label="clean")
        ax.plot(xs, y_hat, marker="x", linestyle="--", color="C1", label="predicted")
        ax.grid(True, alpha=0.3)
        if row == 0:
            ax.set_title(f"c={col}")
        if col == 0:
            ax.set_ylabel(name.upper())
        if row == len(MODEL_NAMES) - 1:
            ax.set_xlabel("t")
        if row == 0 and col == 0:
            ax.legend(loc="best", fontsize=8)
fig.suptitle("F19: predicted vs. clean window — first-seed models")
fig.tight_layout()
plt.show()


## 6. F18 — Noise-robustness curve

For each first-seed checkpoint, evaluate test MSE on a freshly noised
mixed signal across a range of σ values. Models trained at σ=0.5
should degrade gracefully as noise increases.

In [ ]:
sigmas = [0.0, 0.1, 0.25, 0.5, 0.75, 1.0, 1.5]

robust_curves: dict[str, np.ndarray] = {}
data_cfg = BASE_CONFIG["data"]
for name in MODEL_NAMES:
    sdk = first_seed_sdks[name]
    run = first_seed_runs[name]
    pure = sdk.generate_data().pure
    result = sweep_noise(
        run.model,
        pure,
        sigmas,
        context_window=int(data_cfg["context_window"]),
        split=data_cfg["split"],
        batch_size=int(BASE_CONFIG["training"]["batch_size"]),
        seed=int(data_cfg["data_seed"]),
    )
    robust_curves[name] = result.mses
    print(f"{name}: " + ", ".join(
        f"σ={s:.2f}→{m:.3f}" for s, m in zip(result.sigmas, result.mses)
    ))

fig, ax = plt.subplots(figsize=(6.5, 4.0))
for name in MODEL_NAMES:
    ax.plot(sigmas, robust_curves[name], marker="o", label=name.upper())
ax.set_xlabel("noise σ at evaluation time")
ax.set_ylabel("test MSE")
ax.set_yscale("log")
ax.set_title("F18: noise-robustness — test MSE vs. σ")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

plot_sweep_line(
    sigmas,
    np.stack([robust_curves[n] for n in MODEL_NAMES]),
    out_path=NOTEBOOK_OUT / "robustness_sigma.png",
    axis_name="noise σ",
    series_labels=[n.upper() for n in MODEL_NAMES],
    title="F18: noise-robustness sweep",
)


## 7. G3 — Hyperparameter sweeps (≥3 axes)

For each model, sweep three axes one at a time, holding the rest at
their notebook defaults. Each sweep cell writes a PNG under
`runs/notebook/sweeps/` via `plotting.sweep.plot_sweep_line`.

Axes:
1. **hidden size** — capacity.
2. **learning rate** — optimisation regime.
3. **batch size** — gradient noise / steps per epoch.

In [ ]:
def _override_model(cfg: dict, model_name: str, **kwargs) -> dict:
    new_cfg = copy.deepcopy(cfg)
    new_cfg["model"][model_name] = {**new_cfg["model"][model_name], **kwargs}
    return new_cfg


def _override_training(cfg: dict, **kwargs) -> dict:
    new_cfg = copy.deepcopy(cfg)
    new_cfg["training"] = {**new_cfg["training"], **kwargs}
    return new_cfg


def _train_eval(cfg: dict, model_name: str, seed: int = 0) -> float:
    sdk = SDK(config=cfg)
    run = sdk.train(model_name, seed=seed)
    return sdk.evaluate(run).test_mse


sweep_root = NOTEBOOK_OUT / "sweeps"
sweep_root.mkdir(parents=True, exist_ok=True)


In [ ]:
hidden_values = [8, 16, 32, 64]
hidden_results: dict[str, list[float]] = {}
for name in MODEL_NAMES:
    row: list[float] = []
    for h in hidden_values:
        cfg = _override_model(BASE_CONFIG, name, hidden_size=h)
        row.append(_train_eval(cfg, name))
    hidden_results[name] = row
    print(f"{name}: hidden_size sweep = " + ", ".join(
        f"{h}→{m:.3f}" for h, m in zip(hidden_values, row)
    ))

plot_sweep_line(
    hidden_values,
    np.stack([hidden_results[n] for n in MODEL_NAMES]),
    out_path=sweep_root / "hidden_size.png",
    axis_name="hidden size",
    series_labels=[n.upper() for n in MODEL_NAMES],
    title="G3: hidden size sweep",
)

fig, ax = plt.subplots(figsize=(6.0, 3.8))
for name in MODEL_NAMES:
    ax.plot(hidden_values, hidden_results[name], marker="o", label=name.upper())
ax.set_xlabel("hidden size")
ax.set_ylabel("test MSE")
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_title("G3 axis 1 — hidden size")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
lr_values = [1e-4, 3e-4, 1e-3, 3e-3]
lr_results: dict[str, list[float]] = {}
for name in MODEL_NAMES:
    row: list[float] = []
    for lr in lr_values:
        cfg = _override_training(BASE_CONFIG, lr=lr)
        row.append(_train_eval(cfg, name))
    lr_results[name] = row
    print(f"{name}: lr sweep = " + ", ".join(
        f"{lr:g}→{m:.3f}" for lr, m in zip(lr_values, row)
    ))

plot_sweep_line(
    lr_values,
    np.stack([lr_results[n] for n in MODEL_NAMES]),
    out_path=sweep_root / "learning_rate.png",
    axis_name="learning rate",
    series_labels=[n.upper() for n in MODEL_NAMES],
    title="G3: learning rate sweep",
)

fig, ax = plt.subplots(figsize=(6.0, 3.8))
for name in MODEL_NAMES:
    ax.plot(lr_values, lr_results[name], marker="o", label=name.upper())
ax.set_xlabel("learning rate")
ax.set_ylabel("test MSE")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("G3 axis 2 — learning rate")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
batch_values = [32, 64, 128, 256]
batch_results: dict[str, list[float]] = {}
for name in MODEL_NAMES:
    row: list[float] = []
    for bs in batch_values:
        cfg = _override_training(BASE_CONFIG, batch_size=bs)
        row.append(_train_eval(cfg, name))
    batch_results[name] = row
    print(f"{name}: batch_size sweep = " + ", ".join(
        f"{bs}→{m:.3f}" for bs, m in zip(batch_values, row)
    ))

plot_sweep_line(
    batch_values,
    np.stack([batch_results[n] for n in MODEL_NAMES]),
    out_path=sweep_root / "batch_size.png",
    axis_name="batch size",
    series_labels=[n.upper() for n in MODEL_NAMES],
    title="G3: batch size sweep",
)

fig, ax = plt.subplots(figsize=(6.0, 3.8))
for name in MODEL_NAMES:
    ax.plot(batch_values, batch_results[name], marker="o", label=name.upper())
ax.set_xlabel("batch size")
ax.set_ylabel("test MSE")
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_title("G3 axis 3 — batch size")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()


## 8. Summary

The cells above cover every PRD §A5 item:

| PRD ref | Section |
|---------|---------|
| G2  | §3 — bar chart of mean ± std test MSE per model |
| F17 | §4 — per-component MSE table + grouped bar chart |
| F19 | §5 — predicted-vs-clean grid (model × component) |
| F18 | §6 — log-scale test MSE vs. σ curve |
| G3  | §7 — three sweep plots (hidden size, lr, batch size) |

Re-running `Restart & Run All` reproduces every figure end-to-end. For
larger-scale numbers, run the training CLI on the production config
(`uv run python -m sine_denoiser.train --config config/default.json`).
